In [18]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

In [19]:
# Reading path

melb_data_path = "melb_data.csv"

melb_read_data = pd.read_csv(melb_data_path)

melb_read_data.head()


,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


In [20]:
# Train set of data

y = melb_read_data.Price

features = ["Rooms", "Bathroom", "Landsize", "BuildingArea", "Regionname",]

X = melb_read_data[features]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=1)



In [21]:
# Missing value check Imputation

# from sklearn.impute import SimpleImputer

# imputer = SimpleImputer()

# imputed_X_train = pd.DataFrame(imputer.fit_transform(X_train))
# imputed_X_valid = pd.DataFrame(imputer.transform(X_valid))

# imputed_X_train.columns = X_train.columns
# imputed_X_valid.columns = X_valid.columns

In [22]:
# Get Categorical variables

# var = (X_train.dtypes == "str")
# object_cols = list(var[var].index)

object_cols = list(X_train.select_dtypes(include=["object", "str"]))

print("Categorical Variable")
print(object_cols)

Categorical Variable
['Regionname']


In [23]:
# Drop of categorical variable

drop_X_train = X_train.select_dtypes(exclude=["str"])
drop_X_valid = X_valid.select_dtypes(exclude=["str"])

In [24]:
# Ordinal method

from sklearn.preprocessing import OrdinalEncoder

label_X_train = X_train.copy()
label_X_valid = X_valid.copy()

encoder = OrdinalEncoder()

label_X_train[object_cols] = encoder.fit_transform(X_train[object_cols])
label_X_valid[object_cols] = encoder.transform(X_valid[object_cols])

In [25]:
# One Hot Encoding for preprocessing

from sklearn.preprocessing import OneHotEncoder

OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(X_train[object_cols]))
OH_cols_valid = pd.DataFrame(OH_encoder.transform(X_valid[object_cols]))

OH_cols_train.index = X_train.index
OH_cols_valid.index = X_valid.index

num_X_train = X_train.drop(object_cols, axis=1)
num_X_valid = X_valid.drop(object_cols, axis=1)

OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)

OH_X_train.columns = OH_X_train.columns.astype("str")
OH_X_valid.columns = OH_X_valid.columns.astype("str")

In [26]:
model = RandomForestRegressor(n_estimators=100, random_state=1)

model.fit(OH_X_train, y_train)

val_predicts = model.predict(OH_X_valid)

method = (mean_absolute_error(y_valid, val_predicts))

print(method)


304609.1253874212
